# 02. YOLOv8n → Hailo-8L HEF (Dataflow Compiler)

PC(WSL2 Ubuntu 22.04)의 `hailo_env` 가상환경에서 실행합니다.

1. `01_export_yolov8n.py`로 만든 TFLite 모델을 파싱합니다. NMS는 Hailo에서 레이어로 지원하지 않아 NMS 앞 노드에서 잘라냅니다.
2. COCO2017 train에서 1,000장을 뽑아 보정 데이터셋을 만들고, 입력 정규화와 NMS 후처리를 model script로 붙여 정수 양자화(PTQ)합니다.
3. Hailo-8L용 HEF로 컴파일합니다.


In [ ]:
# First of all, you should run the code in below
# sudo apt-get install -y libgl1-mesa-dev
# in jupyter notebook,
# !sudo apt-get install -y libgl1-mesa-dev

import hailo_sdk_client
from hailo_sdk_client import ClientRunner
ACCELERATOR_CHIP_MODEL = 'hailo8l'

In [ ]:
!mkdir ./COCO2017

!wget http://images.cocodataset.org/zips/train2017.zip -O ./COCO2017/train2017.zip
!unzip ./COCO2017/train2017.zip -d ./COCO2017/train2017
!rm ./COCO2017/train2017.zip

!wget http://images.cocodataset.org/zips/val2017.zip -O ./COCO2017/val2017.zip
!unzip ./COCO2017/val2017.zip -d ./COCO2017/val2017
!rm ./COCO2017/val2017.zip

!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O ./COCO2017/annotations_trainval2017.zip
!unzip ./COCO2017/annotations_trainval2017.zip -d ./COCO2017/annotations_trainval2017
!rm ./COCO2017/annotations_trainval2017.zip

In [ ]:
MODEL_PATH = pathlib.Path('./tf_model')
model_list = []
for current_model_dir in MODEL_PATH.iterdir():
    for current_model_path in current_model_dir.glob('*_float32.tflite'):
        model_list.append(current_model_path)

print(model_list)

In [ ]:
SEARCH_TERM = 'yolov8n'
TARGET_MODEL_INDEX = None
for idx, path in enumerate(model_list):
    if SEARCH_TERM in str(path):
        TARGET_MODEL_INDEX = idx
        print(f"'{SEARCH_TERM}'이 포함된 인덱스는 {TARGET_MODEL_INDEX}입니다.")
        break

print(model_list[TARGET_MODEL_INDEX].stem)

In [ ]:
# yolov8n
# model_284/tf.math.add_1758/Add;model_284/tf.nn.convolution_1410/convolution
# model_284/tf.math.add_1761/Add;model_284/tf.nn.convolution_1413/convolution
# model_284/tf.math.add_1771/Add;model_284/tf.nn.convolution_1423/convolution
# model_284/tf.math.add_1772/Add;model_284/tf.nn.convolution_1424/convolution
# model_284/tf.math.add_1780/Add;model_284/tf.nn.convolution_1432/convolution
# model_284/tf.math.add_1781/Add;model_284/tf.nn.convolution_1433/convolution

# yolov8m
# model_316/tf.math.add_1942/Add;model_316/tf.nn.convolution_1556/convolution
# model_316/tf.math.add_1943/Add;model_316/tf.nn.convolution_1557/convolution
# model_316/tf.math.add_1956/Add;model_316/tf.nn.convolution_1570/convolution
# model_316/tf.math.add_1955/Add;model_316/tf.nn.convolution_1569/convolution
# model_316/tf.math.add_1966/Add;model_316/tf.nn.convolution_1580/convolution
# model_316/tf.math.add_1967/Add;model_316/tf.nn.convolution_1581/convolution

HAILO_MODEL_PROCESS_PATH = pathlib.Path('./hailo_model')
HAILO_MODEL_PROCESS_PATH.mkdir(exist_ok = True)

runner = hailo_sdk_client.ClientRunner(hw_arch = ACCELERATOR_CHIP_MODEL)
hn, npz = runner.translate_tf_model(
        model_path = model_list[TARGET_MODEL_INDEX],
        net_name = model_list[TARGET_MODEL_INDEX].stem,
        # start_node_names = [ # Killed for test
        #     ''
        # ],
        end_node_names = [
            'model_284/tf.math.add_1758/Add;model_284/tf.nn.convolution_1410/convolution',
            'model_284/tf.math.add_1761/Add;model_284/tf.nn.convolution_1413/convolution',
            'model_284/tf.math.add_1771/Add;model_284/tf.nn.convolution_1423/convolution',
            'model_284/tf.math.add_1772/Add;model_284/tf.nn.convolution_1424/convolution',
            'model_284/tf.math.add_1780/Add;model_284/tf.nn.convolution_1432/convolution',
            'model_284/tf.math.add_1781/Add;model_284/tf.nn.convolution_1433/convolution'
        ],
        # tensor_shapes = { # Killed for test
        #     '': [1, 3, 640, 640]
        # }
    )


runner.save_har(
    har_path = str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem +'.har'))
)
!hailo visualizer {str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem +'.har'))} --out-path {str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem +'.svg'))} --no-browser

import IPython
import IPython.display

IPython.display.SVG('./' + str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem +'.svg')))

# Compare with Non Maximum Supress(NMS postprocess) configuration at below repository
# https://github.com/hailo-ai/hailo_model_zoo/tree/master/hailo_model_zoo/cfg/postprocess_config
# e.g.: hailo_model_zoo/cfg/postprocess_config/yolov8n_nms_confg.json

## 최적화(PTQ) 준비

- 보정 데이터는 학습 데이터와 같은 분포여야 해서, YOLOv8n의 사전학습 데이터인 COCO2017 train에서 뽑았습니다.
- 입력 정규화는 model script의 `normalization` 레이어로 모델 안에 넣었습니다. 그래서 보정 데이터와 추론 입력 모두 정규화하지 않은 RGB를 그대로 넣습니다.
- 흑백 이미지는 전처리에서 예외가 나서 건너뜁니다.


In [ ]:

# ImageNet Norm. value: mean = [123.675, 116.28, 103.53], var = [58.395, 57.12, 57.375]
# 자연물, RGB 3원광에 대한 Dataset인 경우 위의 수치를 따르는 것이 권고됨
import tensorflow as tf
from tensorflow.python.eager.context import eager_mode

# -----------------------------------------
# Pre processing (prepare the input images)
# -----------------------------------------
def preproc(image, output_height=640, output_width=640):
    """Preprocess image to fixed size (640x640) with aspect ratio preservation"""
    with eager_mode():
        h, w = image.shape[0], image.shape[1]
        scale = tf.cond( # resize scale target configuration
            pred = tf.less(h, w), # 짧은 방향으로 resize
            true_fn = lambda: output_height / h, 
            false_fn = lambda: output_width / w
        )
        resized_image = tf.compat.v1.image.resize_bilinear(
            tf.expand_dims(image, 0), [int(h * scale), int(w * scale)]
        )
        cropped_image = tf.compat.v1.image.resize_with_crop_or_pad(
            resized_image, output_height, output_width
        )

        return tf.squeeze(cropped_image)


In [ ]:
from pathlib import Path

images_path = Path("./COCO2017/train2017/train2017")
images_list = [img.name for img in images_path.iterdir() if img.suffix == ".jpg"]
print(len(images_list))


In [ ]:
import numpy as np
from PIL import Image
import random
import time

CALIBRATION_SET_SIZE = 1000 # minimum size: 1000
TARGET_MODEL_HEIGHT = 640
TARGET_MODEL_WIDTH = 640

random.seed(time.time())
sampled_images_list = random.sample(images_list, CALIBRATION_SET_SIZE)

calib_dataset = np.zeros((len(sampled_images_list), TARGET_MODEL_HEIGHT, TARGET_MODEL_WIDTH, 3))
for idx, img_name in enumerate(sorted(sampled_images_list)):
    current_img_path = images_path / img_name
    img = np.array(Image.open(current_img_path))
    try:
        img_preproc = preproc(img, output_height = TARGET_MODEL_HEIGHT, output_width = TARGET_MODEL_WIDTH)
    except Exception as ex:
        print(f" {ex}")
        print(f"GRAYSCALE IMAGE SKIP:{img_name}")
        continue
    calib_dataset[idx, :, :, :] = img_preproc.numpy()

mean_per_channel = np.mean(calib_dataset, axis=(0, 1, 2))
std_per_channel = np.std(calib_dataset, axis=(0, 1, 2))

print("Calculated mean per channel:", mean_per_channel)
print("Calculated std per channel:", std_per_channel)

np.save("calib_set_random_COCO2017_train.npy", calib_dataset)

In [ ]:
script_alls = [
    f"normalization1 = normalization({[round(value, 2) for value in mean_per_channel]}, {[round(value, 2) for value in std_per_channel]})\n",
    "nms_postprocess(meta_arch=yolov8, engine=cpu, nms_scores_th=0.2, nms_iou_th=0.4)\n"
]

runner.load_model_script("".join(script_alls))
runner.optimize(calib_dataset)
runner.save_har(
    har_path = str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem +'_optimized')) + '.har'
)

In [ ]:
hef = runner.compile()

file_name = str(HAILO_MODEL_PROCESS_PATH/(model_list[TARGET_MODEL_INDEX].stem)) + '.hef'
with open(file_name, "wb") as f:
    f.write(hef)